# 13 — Version CORRIGÉE du pipeline graphe (pour Cissokho)

Autonome (aucun import maison), runnable sur Kaggle/Colab/local. Corrige les bugs de D2026 :
- ❌ supprime les features dérivées de `fraud_flag` (fuite) ; ✅ historique compte en **TE fold-safe**
- ✅ **validation temporelle** (pas de `train_test_split` aléatoire)
- ✅ métrique **Average Precision** (pas ROC AUC)
- ✅ restriction **op_03** ; ✅ **pas de calibration**
- garde l'angle **graphe** (degrés in/out + agrégation de voisinage à 1 saut), sans GPU

Sortie : `oof_graph.csv` (train op_03) + `test_graph.csv` (test) -> à blender avec le CatBoost.

In [ ]:
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import average_precision_score

# adapte le chemin :
DATA = "../data"   # Kaggle : '/kaggle/input/.../'
TARGET = "fraud_flag"; ID = "id"; PERIOD = "period"; OP = "operation"; AMT = "amount"
OACC, DACC = "origin_account", "destination_account"
OBB, OBA = "origin_balance_before", "origin_balance_after"
DBB, DBA = "destination_balance_before", "destination_balance_after"
FRAUD_OP = "op_03"
DEVICE = "cpu"   # Kaggle GPU : 'cuda'

train = pd.read_csv(f"{DATA}/train.csv")
test = pd.read_csv(f"{DATA}/test.csv")
print("train", train.shape, "| test", test.shape)

In [ ]:
# ---- Validation temporelle : fenêtre expansive sur les périodes ----
def time_folds(period, n_splits=5):
    period = pd.Series(period).reset_index(drop=True)
    uniq = np.sort(period.unique())
    blocks = np.array_split(uniq, n_splits + 1)
    for i in range(1, n_splits + 1):
        tr = period.index[period.isin(np.concatenate(blocks[:i]))].to_numpy()
        va = period.index[period.isin(blocks[i])].to_numpy()
        yield tr, va

# ---- Target encoding compte émetteur, SANS FUITE ----
def fit_target_map(ref, col, smoothing=30):
    gm = ref[TARGET].mean()
    agg = ref.groupby(col)[TARGET].agg(["mean", "count"])
    smooth = (agg["mean"] * agg["count"] + gm * smoothing) / (agg["count"] + smoothing)
    return smooth.to_dict(), gm

def oof_te(train_df, col, smoothing=30, n_splits=5, seed=42):
    out = np.zeros(len(train_df)); vals = train_df[col].to_numpy()
    for tr, va in KFold(n_splits, shuffle=True, random_state=seed).split(train_df):
        mp, gm = fit_target_map(train_df.iloc[tr], col, smoothing)
        out[va] = pd.Series(vals[va]).map(mp).fillna(gm).to_numpy()
    return out

In [ ]:
# ---- Features graphe + comportementales, TOUTES apprises sur `ref` (= passé) ----
EPS = 1e-6
def graph_feats(df, ref):
    df = df.reset_index(drop=True)
    f = pd.DataFrame(index=df.index)
    # row-level (aucune fuite, dispo au scoring)
    f["amount_log1p"] = np.log1p(np.maximum(df[AMT], 0))
    f["amt_vs_obb"] = df[AMT] / (df[OBB].abs() + EPS)
    f["amt_vs_dbb"] = df[AMT] / (df[DBB].abs() + EPS)
    f["obb"] = df[OBB]; f["dbb"] = df[DBB]
    f["o_resid"] = df[OBA] - (df[OBB] - df[AMT])
    f["d_resid"] = df[DBA] - (df[DBB] + df[AMT])
    # fréquences (comptage sur le passé)
    fo = ref[OACC].value_counts(normalize=True); fd = ref[DACC].value_counts(normalize=True)
    f["freq_o"] = df[OACC].map(fo).fillna(0).values
    f["freq_d"] = df[DACC].map(fd).fillna(0).values
    # degrés bipartites (graphe sur le passé)
    o_out = ref.groupby(OACC)[DACC].nunique()      # fan-out émetteur
    d_in = ref.groupby(DACC)[OACC].nunique()       # collecteur destinataire
    o_cnt = ref[OACC].value_counts(); d_cnt = ref[DACC].value_counts()
    f["o_out_deg"] = df[OACC].map(o_out).fillna(0).values
    f["d_in_deg"] = df[DACC].map(d_in).fillna(0).values
    f["o_cnt"] = df[OACC].map(o_cnt).fillna(0).values
    f["d_cnt"] = df[DACC].map(d_cnt).fillna(0).values
    # agrégation de voisinage à 1 saut : degré entrant moyen des destinataires de l'émetteur
    ref2 = ref[[OACC, DACC]].copy(); ref2["d_in"] = ref2[DACC].map(d_in)
    o_nbr = ref2.groupby(OACC)["d_in"].mean()
    f["o_nbr_mean_d_in"] = df[OACC].map(o_nbr).fillna(0).values
    return f

def make_xgb():
    return xgb.XGBClassifier(objective="binary:logistic", eval_metric="aucpr",
                             max_depth=6, learning_rate=0.05, n_estimators=600,
                             subsample=0.8, colsample_bytree=0.8, random_state=42,
                             tree_method="hist", device=DEVICE, n_jobs=-1)

## CV temporelle sur op_03 (métrique = Average Precision)

In [ ]:
op03 = (train[OP] == FRAUD_OP).to_numpy()
y = train[TARGET].to_numpy()
folds = list(time_folds(train[PERIOD]))
oof = np.zeros(len(train)); per_fold = []
for tr_idx, va_idx in folds:
    tr_op = tr_idx[op03[tr_idx]]; va_op = va_idx[op03[va_idx]]
    ref = train.iloc[tr_op]
    Xtr = graph_feats(train.iloc[tr_op], ref); Xtr["te_origin"] = oof_te(ref, OACC)
    Xva = graph_feats(train.iloc[va_op], ref)
    mp, gm = fit_target_map(ref, OACC); Xva["te_origin"] = train.iloc[va_op][OACC].map(mp).fillna(gm).values
    m = make_xgb().fit(Xtr, y[tr_op])
    oof[va_op] = m.predict_proba(Xva)[:, 1]
    per_fold.append(average_precision_score(y[va_op], oof[va_op]))
    print(f"fold AP = {per_fold[-1]:.4f}")
print(f"\nAP graphe : global {np.mean(per_fold):.4f} | recent(2) {np.mean(per_fold[-2:]):.4f} | last {per_fold[-1]:.4f}")

## Sorties pour le blend : OOF (train op_03) + prédictions test

In [ ]:
# OOF train op_03 (non calibré) -> pour optimiser le poids du blend
oof_df = train.loc[op03, [ID]].copy(); oof_df["oof"] = oof[op03]
oof_df.to_csv("oof_graph.csv", index=False)

# Modèle final sur tout le train op_03, prédiction test (0 hors op_03)
ref_full = train[op03]
Xf = graph_feats(ref_full, ref_full); Xf["te_origin"] = oof_te(ref_full, OACC)
mfin = make_xgb().fit(Xf, y[op03])
te_op = (test[OP] == FRAUD_OP).to_numpy()
Xte = graph_feats(test[te_op], ref_full)
mp, gm = fit_target_map(ref_full, OACC); Xte["te_origin"] = test[te_op][OACC].map(mp).fillna(gm).values
proba = mfin.predict_proba(Xte)[:, 1]
full = np.zeros(len(test)); full[te_op] = proba
sub = pd.DataFrame({ID: test[ID], "target": full})
sub.to_csv("test_graph.csv", index=False)
print("écrit : oof_graph.csv (", len(oof_df), ") et test_graph.csv (", len(sub), ")")
print("-> envoie ces 2 fichiers pour le blend avec le CatBoost")